# Imports

In [594]:
from math import log as log
from math import inf as inf
from math import floor as floor
import numpy as np

# Constantes Globais

In [595]:
p = 0.5
N = 500 #definido arbitrariamente
MAXLEVEL = int(log(N, 1/p))

# TASKs

In [596]:
class Task:
    def __init__(self, atributos: dict):
        #att = ["task_id", "start_time", "end_times"]
        self.atributos = atributos

    def __getitem__(self, chave):
        return self.atributos[chave]

    def __setitem__(self, chave, valor):
        self.atributos[chave] = valor

# Skip List - ATIVIDADE B)

In [597]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [598]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [599]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca
    y = self.cabeca
    output = [None]*(self.tam+1)
    nvly = self.nivel_atual

    for i in range(self.nivel_atual, -1, -1):
      while y.frente[i] != self.NULL and y.frente[i].conteudo['task_id'] <= chaveBuscaB:
        y = y.frente[i]
      nvly = i

    for j in range(nvly,-1,-1):
      while x.frente[j] != self.NULL and x.frente[j].conteudo['task_id'] < chaveBuscaA:
        x = x.frente[j]

    x = x.frente[0]
    c = 0

    if x.conteudo['task_id'] >= chaveBuscaA and y.conteudo['task_id'] <= chaveBuscaB:
      while x.conteudo['task_id'] <= y.conteudo['task_id']:
        output[c] = x.conteudo
        x = x.frente[0]
        c+=1

    return output

## EXEMPLO DE USO - SKIP LIST

In [600]:
skip = Skip_list()

### TESTE INSERT

In [601]:
#c = ["task_id", "start_time", "end_times"]
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(0,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  skip.insert(x,y,z)


### VISUALIZAÇÃO SKIPLIST

In [602]:
def imprime_linear(skip:Skip_list):
  x = skip.cabeca.frente[0]
  while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

In [603]:
def imprime_estrutura(skip:Skip_list):
  x = skip.cabeca.frente[0]
  for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

In [604]:
imprime_linear(skip)

{'task_id': 0, 'start_time': 90, 'end_time': 301}
{'task_id': 1, 'start_time': 943, 'end_time': 1271}
{'task_id': 26, 'start_time': 146, 'end_time': 1116}
{'task_id': 31, 'start_time': 859, 'end_time': 1206}
{'task_id': 45, 'start_time': 101, 'end_time': 1728}
{'task_id': 62, 'start_time': 185, 'end_time': 1571}
{'task_id': 78, 'start_time': 20, 'end_time': 526}
{'task_id': 84, 'start_time': 235, 'end_time': 275}


In [605]:
imprime_estrutura(skip)

--- Nível 6 ---
26 -> NULL
--- Nível 5 ---
26 -> NULL
--- Nível 4 ---
26 -> NULL
--- Nível 3 ---
26 -> NULL
--- Nível 2 ---
26 -> NULL
--- Nível 1 ---
0 -> 1 -> 26 -> 45 -> 62 -> 84 -> NULL
--- Nível 0 ---
0 -> 1 -> 26 -> 31 -> 45 -> 62 -> 78 -> 84 -> NULL


## TESTE DELETE

In [606]:
skip.delete(5)

### VISUALIZAÇÃO SKIPLIST

In [607]:
imprime_linear(skip)

{'task_id': 0, 'start_time': 90, 'end_time': 301}
{'task_id': 1, 'start_time': 943, 'end_time': 1271}
{'task_id': 26, 'start_time': 146, 'end_time': 1116}
{'task_id': 31, 'start_time': 859, 'end_time': 1206}
{'task_id': 45, 'start_time': 101, 'end_time': 1728}
{'task_id': 62, 'start_time': 185, 'end_time': 1571}
{'task_id': 78, 'start_time': 20, 'end_time': 526}
{'task_id': 84, 'start_time': 235, 'end_time': 275}


In [608]:
imprime_estrutura(skip)

--- Nível 6 ---
26 -> NULL
--- Nível 5 ---
26 -> NULL
--- Nível 4 ---
26 -> NULL
--- Nível 3 ---
26 -> NULL
--- Nível 2 ---
26 -> NULL
--- Nível 1 ---
0 -> 1 -> 26 -> 45 -> 62 -> 84 -> NULL
--- Nível 0 ---
0 -> 1 -> 26 -> 31 -> 45 -> 62 -> 78 -> 84 -> NULL


## Teste SEARCH

In [609]:
skip.search(26)

True

### VISUALIZAÇÃO SKIPLIST

In [610]:
imprime_linear(skip)

{'task_id': 0, 'start_time': 90, 'end_time': 301}
{'task_id': 1, 'start_time': 943, 'end_time': 1271}
{'task_id': 26, 'start_time': 146, 'end_time': 1116}
{'task_id': 31, 'start_time': 859, 'end_time': 1206}
{'task_id': 45, 'start_time': 101, 'end_time': 1728}
{'task_id': 62, 'start_time': 185, 'end_time': 1571}
{'task_id': 78, 'start_time': 20, 'end_time': 526}
{'task_id': 84, 'start_time': 235, 'end_time': 275}


In [611]:
imprime_estrutura(skip)

--- Nível 6 ---
26 -> NULL
--- Nível 5 ---
26 -> NULL
--- Nível 4 ---
26 -> NULL
--- Nível 3 ---
26 -> NULL
--- Nível 2 ---
26 -> NULL
--- Nível 1 ---
0 -> 1 -> 26 -> 45 -> 62 -> 84 -> NULL
--- Nível 0 ---
0 -> 1 -> 26 -> 31 -> 45 -> 62 -> 78 -> 84 -> NULL


## Teste SEARCHINTERVAL

In [612]:
r = skip.searchinterval(0,0)

In [613]:
for i in range(len(r)):
  if r[i] != None:
    print(r[i].atributos)

{'task_id': 0, 'start_time': 90, 'end_time': 301}


### VISUALIZAÇÃO SKIPLIST

In [614]:
imprime_linear(skip)

{'task_id': 0, 'start_time': 90, 'end_time': 301}
{'task_id': 1, 'start_time': 943, 'end_time': 1271}
{'task_id': 26, 'start_time': 146, 'end_time': 1116}
{'task_id': 31, 'start_time': 859, 'end_time': 1206}
{'task_id': 45, 'start_time': 101, 'end_time': 1728}
{'task_id': 62, 'start_time': 185, 'end_time': 1571}
{'task_id': 78, 'start_time': 20, 'end_time': 526}
{'task_id': 84, 'start_time': 235, 'end_time': 275}


In [615]:
imprime_estrutura(skip)

--- Nível 6 ---
26 -> NULL
--- Nível 5 ---
26 -> NULL
--- Nível 4 ---
26 -> NULL
--- Nível 3 ---
26 -> NULL
--- Nível 2 ---
26 -> NULL
--- Nível 1 ---
0 -> 1 -> 26 -> 45 -> 62 -> 84 -> NULL
--- Nível 0 ---
0 -> 1 -> 26 -> 31 -> 45 -> 62 -> 78 -> 84 -> NULL


# AVL - ATIVIDADE C)

In [616]:
class No_AVL:
  def __init__(self, esq=None, dir=None, task:Task=None):
    self.esq = esq
    self.dir = dir
    self.fator_balanceamento = 0
    self.altura = 0
    self.conteudo = task

In [617]:
class AVL:
  def __init__(self):
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir

    if n_aux == None:
      return raiz

    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['task_id']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    if chaveBuscaA <= raiz.conteudo['task_id'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['task_id']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    return output



## EXEMPLO DE USO - AVL

In [618]:
AVL = AVL()

In [619]:
raiz = None
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  raiz=AVL.insert(raiz,t)

### Teste INSERT

In [620]:
def percursoIN_ORDER(raiz):
  if raiz == None:
    return
  percursoIN_ORDER(raiz.esq)
  print(raiz.conteudo.atributos)
  percursoIN_ORDER(raiz.dir)

### Visualização

In [621]:
percursoIN_ORDER(raiz)

{'task_id': 20, 'start_time': 880, 'end_time': 1010}
{'task_id': 30, 'start_time': 314, 'end_time': 1131}
{'task_id': 34, 'start_time': 81, 'end_time': 1182}
{'task_id': 52, 'start_time': 689, 'end_time': 1284}
{'task_id': 55, 'start_time': 237, 'end_time': 1181}
{'task_id': 70, 'start_time': 771, 'end_time': 1654}
{'task_id': 74, 'start_time': 125, 'end_time': 1932}
{'task_id': 96, 'start_time': 579, 'end_time': 1374}


### Teste DELETE

In [622]:
raiz=AVL.delete(raiz, 79)

### Visualização

In [623]:
percursoIN_ORDER(raiz)

{'task_id': 20, 'start_time': 880, 'end_time': 1010}
{'task_id': 30, 'start_time': 314, 'end_time': 1131}
{'task_id': 34, 'start_time': 81, 'end_time': 1182}
{'task_id': 52, 'start_time': 689, 'end_time': 1284}
{'task_id': 55, 'start_time': 237, 'end_time': 1181}
{'task_id': 70, 'start_time': 771, 'end_time': 1654}
{'task_id': 74, 'start_time': 125, 'end_time': 1932}
{'task_id': 96, 'start_time': 579, 'end_time': 1374}


### Teste SEARCH

In [624]:
busca=AVL.search(raiz,67)

In [625]:
if busca:
  print(busca.conteudo.atributos)
else:
  print("FALSO")

FALSO


## Teste SEARCHINTERVAL

In [626]:
output = []

In [627]:
buscaIntervalo = AVL.searchinterval(raiz,0,60, output)
for i in buscaIntervalo:
  print(i.conteudo.atributos)

{'task_id': 20, 'start_time': 880, 'end_time': 1010}
{'task_id': 30, 'start_time': 314, 'end_time': 1131}
{'task_id': 34, 'start_time': 81, 'end_time': 1182}
{'task_id': 52, 'start_time': 689, 'end_time': 1284}
{'task_id': 55, 'start_time': 237, 'end_time': 1181}


# ATIVIDADE - D)
Adapted from: https://github.com/amitjoshi2724/gpi-job-scheduling
## corpus sintético

In [628]:
import random

## Geração de Tasks - Especificação da Atividade

In [629]:
def generate_uniform_tasks(n):

    MAX_VAL = 10**6
    tasks = [(random.randint(0, MAX_VAL), random.randint(0, MAX_VAL), id) for id in range(n)]
    tasks = [{'task_id': id,
              'start_time': min(s, e),
              'end_time': max(s, e)} for s, e, id in tasks]

    return tasks

#t_uniform = generate_uniform_tasks(100)

In [630]:
def generate_normal_tasks(n):

    K = 10**9
    sigma = K / 10

    # Truncated normal start times in [0, K]
    start_times = np.clip(np.random.normal(loc=K/2, scale=sigma, size=n), 0, K)
    durations = np.random.uniform(1.0, 10**6, size=n)
    end_times = start_times + durations
    tasks = [{'task_id': id,
              'start_time': s,
              'end_time': e} for id, (s, e) in enumerate(zip(start_times, end_times))]

    return tasks

#t_normal = generate_normal_tasks(100)

In [631]:
def generate_zipf_tasks(n):

    K = 10**9  # Upper bound of time domain

    # Clustered start times near 0 using truncated exponential
    scale = K / 10  # Controls clustering tightness; lower = tighter near 0
    start_times = np.random.exponential(scale=scale, size=n)
    start_times = np.clip(start_times, 0, K)  # Ensure within [0, K]

    # Zipf durations, scaled and capped
    raw_zipf = np.random.zipf(a=2.0, size=n)
    durations = np.minimum(100 * raw_zipf, 10**6)
    end_times = start_times + durations

    tasks = [{'task_id': id,
              'start_time': float(s),
              'end_time': float(e)} for id, (s, e) in enumerate(zip(start_times, end_times))]

    return tasks

#t_zipf = generate_zipf_tasks(100)

## Adaptação SearchInterval SKIP LIST
A implementação anterior era mais genérica se baseando apenas no valor da chave ($task_{id}$)

Nesta implementação, adaptei para atender ao *timestamp* ($start\_time \geq chaveBuscaA$ e $chaveBuscaB \geq end\_time$)

Contudo, precisei fazer algumas correções por que a forma anterior funcionava para quando queríamos comparar $chaveBuscaA$ ou $chaveBuscaB$ apenas com $task\_id$

Nesta nova versão, descarto tudo o que usava $y$ (apontador superior para definir fim do intervalo) e uso apenas $x$ para percorrer a estrutura uma única vez e mais outra vez para gerar a saída

In [632]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [633]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [634]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      print(x.conteudo.atributos)
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca.frente[0]
    output = [None]*(self.tam+1)

    c = 0
    while x != self.NULL:
      if chaveBuscaA <= x.conteudo['start_time'] and x.conteudo['end_time'] <= chaveBuscaB:
        output[c] = x.conteudo
        c+=1
      x = x.frente[0]

    return output

### Teste Novo SearchInterval

In [635]:
s = Skip_list()

In [636]:
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  s.insert(x,y,z)

In [637]:
v = s.searchinterval(50,1700)

In [638]:
for i in range(len(v)):
  if v[i] != None:
    print(v[i].atributos)

{'task_id': 2, 'start_time': 497, 'end_time': 1542}
{'task_id': 4, 'start_time': 542, 'end_time': 1623}
{'task_id': 47, 'start_time': 112, 'end_time': 1077}
{'task_id': 50, 'start_time': 597, 'end_time': 1122}
{'task_id': 91, 'start_time': 55, 'end_time': 1352}


#### VISUALIZAÇÃO SKIPLIST

In [639]:
imprime_linear(s)

{'task_id': 2, 'start_time': 497, 'end_time': 1542}
{'task_id': 4, 'start_time': 542, 'end_time': 1623}
{'task_id': 10, 'start_time': 310, 'end_time': 1701}
{'task_id': 46, 'start_time': 73, 'end_time': 1740}
{'task_id': 47, 'start_time': 112, 'end_time': 1077}
{'task_id': 50, 'start_time': 597, 'end_time': 1122}
{'task_id': 91, 'start_time': 55, 'end_time': 1352}


In [640]:
imprime_estrutura(s)

--- Nível 4 ---
47 -> NULL
--- Nível 3 ---
47 -> 50 -> NULL
--- Nível 2 ---
47 -> 50 -> NULL
--- Nível 1 ---
46 -> 47 -> 50 -> 91 -> NULL
--- Nível 0 ---
2 -> 4 -> 10 -> 46 -> 47 -> 50 -> 91 -> NULL


## Adaptações AVL - SearchInterval
A adaptação, assim como na *skip list*, foi o uso de $start\_time$ e $end\_time$ ao invés de $task\_id$

De resto a estrutura do algoritmo se manteve a mesma, apenas mudanças nas comparações do *if*

In [641]:
class AVL:
  def __init__(self):
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir

    if n_aux == None:
      return raiz

    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      print(f"Removendo Nó de chave: {chaveBusca}")
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['start_time']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    if chaveBuscaA <= raiz.conteudo['start_time'] and raiz.conteudo['end_time'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['end_time']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    return output

### Teste Novo SearchInterval

In [642]:
avl = AVL()

In [643]:
root = None
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  root=avl.insert(root,t)

In [644]:
output = []
buscaIntervalo = avl.searchinterval(root,600,1800, output)
for i in buscaIntervalo:
  print(i.conteudo.atributos)

### Visualização

In [645]:
percursoIN_ORDER(root)

{'task_id': 23, 'start_time': 322, 'end_time': 1432}
{'task_id': 44, 'start_time': 553, 'end_time': 1469}
{'task_id': 51, 'start_time': 90, 'end_time': 1803}
{'task_id': 60, 'start_time': 360, 'end_time': 1092}
{'task_id': 70, 'start_time': 754, 'end_time': 1329}
{'task_id': 81, 'start_time': 235, 'end_time': 1899}
{'task_id': 85, 'start_time': 956, 'end_time': 1924}
{'task_id': 96, 'start_time': 448, 'end_time': 1911}


## Constantes

In [646]:
w = np.random.random()
while w < 0.5:
  w = np.random.random()
p = w
N = np.random.randint(1,2**12)
MAXLEVEL = int(log(N, 1/p))

In [647]:
p

0.8694261388654876

In [648]:
N

4044

In [649]:
MAXLEVEL

59

## Distribuição Uniforme - Skip List

### Instanciação

In [650]:
skipList = Skip_list()
arvoreAVL = AVL()

### INSERT

In [651]:
t = generate_uniform_tasks(N)
for k in t:
  skipList.insert(k['task_id'], k['start_time'], k['end_time'])

In [652]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 210736, 'end_time': 316830}
{'task_id': 1, 'start_time': 503190, 'end_time': 967214}
{'task_id': 2, 'start_time': 878820, 'end_time': 947915}
{'task_id': 3, 'start_time': 289437, 'end_time': 679583}
{'task_id': 4, 'start_time': 288262, 'end_time': 542565}
{'task_id': 5, 'start_time': 66292, 'end_time': 282602}
{'task_id': 6, 'start_time': 648883, 'end_time': 697956}
{'task_id': 7, 'start_time': 26567, 'end_time': 28755}
{'task_id': 8, 'start_time': 459446, 'end_time': 943228}
{'task_id': 9, 'start_time': 587807, 'end_time': 663523}
{'task_id': 10, 'start_time': 581425, 'end_time': 722345}
{'task_id': 11, 'start_time': 430893, 'end_time': 925249}
{'task_id': 12, 'start_time': 297238, 'end_time': 509565}
{'task_id': 13, 'start_time': 505715, 'end_time': 590445}
{'task_id': 14, 'start_time': 174736, 'end_time': 303577}
{'task_id': 15, 'start_time': 231530, 'end_time': 233530}
{'task_id': 16, 'start_time': 97434, 'end_time': 729415}
{'task_id': 17, 'start_time'

In [653]:
imprime_estrutura(skipList)

--- Nível 58 ---
1135 -> NULL
--- Nível 57 ---
1135 -> NULL
--- Nível 56 ---
1135 -> NULL
--- Nível 55 ---
1135 -> NULL
--- Nível 54 ---
1135 -> NULL
--- Nível 53 ---
235 -> 1135 -> NULL
--- Nível 52 ---
235 -> 1135 -> NULL
--- Nível 51 ---
235 -> 1135 -> 2538 -> NULL
--- Nível 50 ---
235 -> 1135 -> 2538 -> NULL
--- Nível 49 ---
235 -> 1135 -> 2538 -> NULL
--- Nível 48 ---
235 -> 1135 -> 2538 -> 2882 -> NULL
--- Nível 47 ---
235 -> 1135 -> 2538 -> 2882 -> NULL
--- Nível 46 ---
235 -> 1135 -> 2538 -> 2882 -> NULL
--- Nível 45 ---
235 -> 1135 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 44 ---
235 -> 1004 -> 1135 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 43 ---
235 -> 544 -> 1004 -> 1135 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 42 ---
235 -> 282 -> 504 -> 544 -> 1004 -> 1135 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 41 ---
235 -> 282 -> 504 -> 544 -> 1004 -> 1135 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 40 ---
235 -> 282 -> 504 -> 544 -> 1004 -> 113

### DELETE

In [654]:
skipList.tam

4044

In [655]:
skipList.nivel_atual

58

In [656]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, skipList.tam)
  skipList.delete(z)

In [657]:
skipList.tam

3870

In [658]:
skipList.nivel_atual

53

In [659]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 210736, 'end_time': 316830}
{'task_id': 1, 'start_time': 503190, 'end_time': 967214}
{'task_id': 2, 'start_time': 878820, 'end_time': 947915}
{'task_id': 3, 'start_time': 289437, 'end_time': 679583}
{'task_id': 4, 'start_time': 288262, 'end_time': 542565}
{'task_id': 5, 'start_time': 66292, 'end_time': 282602}
{'task_id': 6, 'start_time': 648883, 'end_time': 697956}
{'task_id': 7, 'start_time': 26567, 'end_time': 28755}
{'task_id': 8, 'start_time': 459446, 'end_time': 943228}
{'task_id': 9, 'start_time': 587807, 'end_time': 663523}
{'task_id': 10, 'start_time': 581425, 'end_time': 722345}
{'task_id': 11, 'start_time': 430893, 'end_time': 925249}
{'task_id': 12, 'start_time': 297238, 'end_time': 509565}
{'task_id': 13, 'start_time': 505715, 'end_time': 590445}
{'task_id': 14, 'start_time': 174736, 'end_time': 303577}
{'task_id': 15, 'start_time': 231530, 'end_time': 233530}
{'task_id': 16, 'start_time': 97434, 'end_time': 729415}
{'task_id': 17, 'start_time'

In [660]:
imprime_estrutura(skipList)

--- Nível 53 ---
235 -> NULL
--- Nível 52 ---
235 -> NULL
--- Nível 51 ---
235 -> 2538 -> NULL
--- Nível 50 ---
235 -> 2538 -> NULL
--- Nível 49 ---
235 -> 2538 -> NULL
--- Nível 48 ---
235 -> 2538 -> 2882 -> NULL
--- Nível 47 ---
235 -> 2538 -> 2882 -> NULL
--- Nível 46 ---
235 -> 2538 -> 2882 -> NULL
--- Nível 45 ---
235 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 44 ---
235 -> 1004 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 43 ---
235 -> 544 -> 1004 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 42 ---
235 -> 282 -> 504 -> 544 -> 1004 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 41 ---
235 -> 282 -> 504 -> 544 -> 1004 -> 1138 -> 2538 -> 2882 -> 3000 -> NULL
--- Nível 40 ---
235 -> 282 -> 504 -> 544 -> 1004 -> 1138 -> 2538 -> 2882 -> 3000 -> 3823 -> NULL
--- Nível 39 ---
65 -> 235 -> 282 -> 504 -> 544 -> 576 -> 1004 -> 1138 -> 2538 -> 2581 -> 2760 -> 2882 -> 3000 -> 3064 -> 3823 -> NULL
--- Nível 38 ---
65 -> 235 -> 282 -> 504 -> 544 -> 576 -> 1004 -> 1112 -> 113

### Search

In [661]:
for i in range(skipList.tam):
  z = np.random.randint(0, skipList.tam)
  if not skipList.search(z):
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {skipList.search(z)}\n=============================================================')

{'task_id': 1219, 'start_time': 479887, 'end_time': 806452}
{'task_id': 713, 'start_time': 12039, 'end_time': 275945}
{'task_id': 3693, 'start_time': 196432, 'end_time': 486390}
{'task_id': 1180, 'start_time': 392224, 'end_time': 410896}
{'task_id': 2814, 'start_time': 433721, 'end_time': 512970}
{'task_id': 496, 'start_time': 175341, 'end_time': 647070}
{'task_id': 1706, 'start_time': 630134, 'end_time': 715731}
{'task_id': 3309, 'start_time': 313852, 'end_time': 884816}
{'task_id': 68, 'start_time': 358741, 'end_time': 439334}
{'task_id': 505, 'start_time': 291521, 'end_time': 293346}
Chave:1768; Resultado da Busca: False
{'task_id': 309, 'start_time': 71746, 'end_time': 570002}
{'task_id': 1950, 'start_time': 260024, 'end_time': 735774}
Chave:2009; Resultado da Busca: False
Chave:366; Resultado da Busca: False
{'task_id': 3631, 'start_time': 350170, 'end_time': 443248}
{'task_id': 637, 'start_time': 341928, 'end_time': 744332}
{'task_id': 1247, 'start_time': 261179, 'end_time': 3612

### SEARCH INTERVAL

In [662]:
for i in range(skipList.tam):
  a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
  b = np.random.randint(0, N*np.random.randint(1,N))
  while a > b:
      a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
      b = np.random.randint(0, N*np.random.randint(1,N))
  r = (skipList.searchinterval(a,b))
  print(f'=============================\nA={a}; B={b}')
  for k in r:
    if k is not None:
      print(k.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
{'task_id': 3085, 'start_time': 572283, 'end_time': 708546}
{'task_id': 3086, 'start_time': 232760, 'end_time': 498859}
{'task_id': 3087, 'start_time': 459005, 'end_time': 659168}
{'task_id': 3088, 'start_time': 242717, 'end_time': 392400}
{'task_id': 3089, 'start_time': 487996, 'end_time': 574804}
{'task_id': 3090, 'start_time': 352895, 'end_time': 412689}
{'task_id': 3091, 'start_time': 319658, 'end_time': 850307}
{'task_id': 3092, 'start_time': 726065, 'end_time': 959517}
{'task_id': 3093, 'start_time': 345312, 'end_time': 474677}
{'task_id': 3096, 'start_time': 647150, 'end_time': 741363}
{'task_id': 3099, 'start_time': 430619, 'end_time': 463981}
{'task_id': 3100, 'start_time': 257808, 'end_time': 863977}
{'task_id': 3101, 'start_time': 383575, 'end_time': 541582}
{'task_id': 3102, 'start_time': 809684, 'end_time': 852481}
{'task_id': 3104, 'start_time': 366988, 'end_time': 833862}
{'task_id': 3105, 'start_time': 688992, '

## Distribuição Normal - Skip List

### Instanciação

In [663]:
skipList = Skip_list()
arvoreAVL = AVL()

### INSERT

In [664]:
t = generate_normal_tasks(N)

for k in t:
  k['start_time'] = k['start_time'].item()
  k['end_time'] = k['end_time'].item()
  skipList.insert(k['task_id'], k['start_time'], k['end_time'])

In [665]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 474606443.6233358, 'end_time': 474721115.2121671}
{'task_id': 1, 'start_time': 520237103.2781001, 'end_time': 520468351.56457514}
{'task_id': 2, 'start_time': 525676880.36815435, 'end_time': 525834039.458694}
{'task_id': 3, 'start_time': 741402245.7601278, 'end_time': 741523344.967388}
{'task_id': 4, 'start_time': 375901029.0748168, 'end_time': 376873146.4883768}
{'task_id': 5, 'start_time': 436726978.1711477, 'end_time': 437681884.2136278}
{'task_id': 6, 'start_time': 382822409.6063163, 'end_time': 382856187.9713731}
{'task_id': 7, 'start_time': 465058379.4471209, 'end_time': 465869738.4044607}
{'task_id': 8, 'start_time': 647538976.6191757, 'end_time': 647858635.8184981}
{'task_id': 9, 'start_time': 549355868.2275599, 'end_time': 549795275.0896754}
{'task_id': 10, 'start_time': 588298429.0357366, 'end_time': 588722272.6131926}
{'task_id': 11, 'start_time': 526138867.6739012, 'end_time': 526143412.9875431}
{'task_id': 12, 'start_time': 468990623.3733797, '

In [666]:
imprime_estrutura(skipList)

--- Nível 52 ---
3564 -> 3766 -> NULL
--- Nível 51 ---
3564 -> 3766 -> NULL
--- Nível 50 ---
3564 -> 3766 -> NULL
--- Nível 49 ---
3564 -> 3766 -> NULL
--- Nível 48 ---
1193 -> 3564 -> 3766 -> NULL
--- Nível 47 ---
1193 -> 3564 -> 3766 -> NULL
--- Nível 46 ---
1193 -> 3564 -> 3766 -> NULL
--- Nível 45 ---
1193 -> 1830 -> 3564 -> 3766 -> NULL
--- Nível 44 ---
1193 -> 1830 -> 2411 -> 3564 -> 3766 -> NULL
--- Nível 43 ---
1193 -> 1830 -> 2411 -> 3564 -> 3766 -> NULL
--- Nível 42 ---
1193 -> 1830 -> 2411 -> 2415 -> 3564 -> 3766 -> 3938 -> NULL
--- Nível 41 ---
1193 -> 1830 -> 2411 -> 2415 -> 3564 -> 3766 -> 3938 -> 4029 -> NULL
--- Nível 40 ---
1193 -> 1830 -> 2411 -> 2415 -> 3460 -> 3564 -> 3766 -> 3938 -> 4029 -> NULL
--- Nível 39 ---
1193 -> 1830 -> 2015 -> 2411 -> 2415 -> 3402 -> 3460 -> 3564 -> 3766 -> 3851 -> 3938 -> 4029 -> NULL
--- Nível 38 ---
1193 -> 1830 -> 2015 -> 2411 -> 2415 -> 3402 -> 3460 -> 3564 -> 3766 -> 3851 -> 3938 -> 4029 -> NULL
--- Nível 37 ---
1193 -> 1220 -> 1830 

### DELETE

In [667]:
skipList.tam

4044

In [668]:
skipList.nivel_atual

52

In [669]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, skipList.tam)
  skipList.delete(z)

In [670]:
skipList.tam

2123

In [671]:
skipList.nivel_atual

52

In [672]:
imprime_linear(skipList)

{'task_id': 1, 'start_time': 520237103.2781001, 'end_time': 520468351.56457514}
{'task_id': 6, 'start_time': 382822409.6063163, 'end_time': 382856187.9713731}
{'task_id': 8, 'start_time': 647538976.6191757, 'end_time': 647858635.8184981}
{'task_id': 13, 'start_time': 504388442.11747456, 'end_time': 505057177.86911404}
{'task_id': 14, 'start_time': 560606921.9666554, 'end_time': 560634676.4719577}
{'task_id': 16, 'start_time': 513440558.2537193, 'end_time': 513646852.2208565}
{'task_id': 17, 'start_time': 592758247.010676, 'end_time': 593382601.9553375}
{'task_id': 21, 'start_time': 431093990.7457465, 'end_time': 431824844.16909826}
{'task_id': 26, 'start_time': 530503614.27028614, 'end_time': 531118481.75718296}
{'task_id': 28, 'start_time': 457820662.1199038, 'end_time': 457859917.93932426}
{'task_id': 31, 'start_time': 543350132.3437775, 'end_time': 544126172.3445494}
{'task_id': 33, 'start_time': 463690082.53436303, 'end_time': 464175019.35332936}
{'task_id': 37, 'start_time': 56563

In [673]:
imprime_estrutura(skipList)

--- Nível 52 ---
3564 -> 3766 -> NULL
--- Nível 51 ---
3564 -> 3766 -> NULL
--- Nível 50 ---
3564 -> 3766 -> NULL
--- Nível 49 ---
3564 -> 3766 -> NULL
--- Nível 48 ---
3564 -> 3766 -> NULL
--- Nível 47 ---
3564 -> 3766 -> NULL
--- Nível 46 ---
3564 -> 3766 -> NULL
--- Nível 45 ---
1830 -> 3564 -> 3766 -> NULL
--- Nível 44 ---
1830 -> 2411 -> 3564 -> 3766 -> NULL
--- Nível 43 ---
1830 -> 2411 -> 3564 -> 3766 -> NULL
--- Nível 42 ---
1830 -> 2411 -> 3564 -> 3766 -> 3938 -> NULL
--- Nível 41 ---
1830 -> 2411 -> 3564 -> 3766 -> 3938 -> 4029 -> NULL
--- Nível 40 ---
1830 -> 2411 -> 3460 -> 3564 -> 3766 -> 3938 -> 4029 -> NULL
--- Nível 39 ---
1830 -> 2411 -> 3460 -> 3564 -> 3766 -> 3851 -> 3938 -> 4029 -> NULL
--- Nível 38 ---
1830 -> 2411 -> 3460 -> 3564 -> 3766 -> 3851 -> 3938 -> 4029 -> NULL
--- Nível 37 ---
1220 -> 1830 -> 2086 -> 2411 -> 3460 -> 3564 -> 3766 -> 3851 -> 3938 -> 4029 -> NULL
--- Nível 36 ---
1220 -> 1830 -> 2086 -> 2411 -> 3460 -> 3564 -> 3730 -> 3766 -> 3839 -> 3851 ->

### Search

In [674]:
for i in range(skipList.tam):
  z = np.random.randint(0, skipList.tam)
  if not skipList.search(z):
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {skipList.search(z)}\n=============================================================')

Chave:853; Resultado da Busca: False
{'task_id': 167, 'start_time': 372591255.07247186, 'end_time': 373063590.1694114}
Chave:945; Resultado da Busca: False
{'task_id': 1394, 'start_time': 503047826.6382752, 'end_time': 504020601.3018768}
Chave:1209; Resultado da Busca: False
Chave:1671; Resultado da Busca: False
{'task_id': 1314, 'start_time': 570444992.1131431, 'end_time': 570888853.9595733}
Chave:1467; Resultado da Busca: False
{'task_id': 2108, 'start_time': 420642738.92473876, 'end_time': 420749063.4713094}
{'task_id': 506, 'start_time': 467303476.0060259, 'end_time': 467980588.538798}
{'task_id': 397, 'start_time': 721150329.1010383, 'end_time': 721755236.7882737}
Chave:445; Resultado da Busca: False
{'task_id': 971, 'start_time': 457958213.36053157, 'end_time': 458074718.6619725}
Chave:1380; Resultado da Busca: False
Chave:1563; Resultado da Busca: False
{'task_id': 1072, 'start_time': 438958920.1658665, 'end_time': 439189766.34325314}
Chave:1675; Resultado da Busca: False
Chave:

### SEARCH INTERVAL

In [675]:
for i in range(skipList.tam):
  a = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  b = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  while a > b:
    a = np.random.randint(N, N*np.random.randint(floor(N/2),N))
    b = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  r = (skipList.searchinterval(a,b))
  print(f'=============================\nA={a}; B={b}')
  for k in r:
    if k is not None:
      print(k.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
A=4356033; B=9069048
A=5402522; B=9364573
A=2565098; B=8913684
A=7120600; B=13975263
A=8762474; B=14250944
A=4523256; B=9785713
A=9211560; B=13888222
A=562134; B=10606281
A=2830622; B=10886166
A=10877586; B=13222225
A=4770073; B=6803685
A=7801796; B=15918400
A=1224229; B=5698865
A=908224; B=8316812
A=7433113; B=8681316
A=4860458; B=5835582
A=238735; B=10262101
A=3495381; B=11406034
A=713464; B=6795366
A=3851518; B=6340145
A=1506988; B=8117337
A=6309162; B=9530797
A=9550973; B=10701648
A=3502866; B=7202518
A=8420178; B=12117410
A=2679340; B=3823808
A=7961129; B=10248523
A=583850; B=4578311
A=1960064; B=12759787
A=4773531; B=6956052
A=10110442; B=13629779
A=7440126; B=12282414
A=141153; B=13377881
A=3801288; B=14669227
A=2203159; B=8438490
A=1626696; B=3116091
A=5244459; B=9530057
A=5695797; B=6490351
A=5817911; B=7131105
A=5097703; B=11152034
A=10085243; B=13836692
A=697546; B=980653
A=3562054; B=14023488
A=1677114; B=2142560
A=

## Distribuição ZIPF - Skip List

### Instanciação

In [676]:
skipList = Skip_list()
arvoreAVL = AVL()

### INSERT

In [677]:
t = generate_zipf_tasks(N)
for k in t:
  skipList.insert(k['task_id'], k['start_time'], k['end_time'])

In [678]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 64373448.79108468, 'end_time': 64373548.79108468}
{'task_id': 1, 'start_time': 474218645.0512261, 'end_time': 474218845.0512261}
{'task_id': 2, 'start_time': 42966948.91698408, 'end_time': 42967048.91698408}
{'task_id': 3, 'start_time': 26571211.8909201, 'end_time': 26571311.8909201}
{'task_id': 4, 'start_time': 298363689.6643726, 'end_time': 298364089.6643726}
{'task_id': 5, 'start_time': 45701643.09563986, 'end_time': 45701943.09563986}
{'task_id': 6, 'start_time': 16906717.39182673, 'end_time': 16906817.39182673}
{'task_id': 7, 'start_time': 33329449.019334782, 'end_time': 33329549.019334782}
{'task_id': 8, 'start_time': 371131990.58638984, 'end_time': 371132090.58638984}
{'task_id': 9, 'start_time': 111384861.80051246, 'end_time': 111385361.80051246}
{'task_id': 10, 'start_time': 13797507.42141199, 'end_time': 13797607.42141199}
{'task_id': 11, 'start_time': 94730012.23914832, 'end_time': 94730112.23914832}
{'task_id': 12, 'start_time': 415806135.634393

In [679]:
imprime_estrutura(skipList)

--- Nível 46 ---
1401 -> NULL
--- Nível 45 ---
1401 -> 2928 -> NULL
--- Nível 44 ---
407 -> 1044 -> 1401 -> 2928 -> 3511 -> NULL
--- Nível 43 ---
407 -> 1044 -> 1401 -> 2668 -> 2928 -> 3439 -> 3511 -> NULL
--- Nível 42 ---
407 -> 1044 -> 1401 -> 2668 -> 2928 -> 3439 -> 3511 -> 3985 -> NULL
--- Nível 41 ---
407 -> 1044 -> 1401 -> 2668 -> 2695 -> 2928 -> 3439 -> 3511 -> 3985 -> NULL
--- Nível 40 ---
206 -> 364 -> 407 -> 1044 -> 1401 -> 1695 -> 2391 -> 2668 -> 2695 -> 2928 -> 3439 -> 3511 -> 3985 -> 4009 -> NULL
--- Nível 39 ---
206 -> 364 -> 407 -> 1044 -> 1401 -> 1695 -> 2167 -> 2391 -> 2668 -> 2695 -> 2928 -> 3439 -> 3511 -> 3871 -> 3985 -> 4009 -> NULL
--- Nível 38 ---
206 -> 364 -> 407 -> 1044 -> 1401 -> 1695 -> 2167 -> 2391 -> 2668 -> 2695 -> 2928 -> 3095 -> 3439 -> 3511 -> 3871 -> 3985 -> 4009 -> NULL
--- Nível 37 ---
206 -> 364 -> 407 -> 1044 -> 1401 -> 1531 -> 1535 -> 1695 -> 2167 -> 2391 -> 2668 -> 2695 -> 2928 -> 3095 -> 3439 -> 3511 -> 3871 -> 3985 -> 3991 -> 4009 -> NULL
--- 

### DELETE

In [680]:
skipList.tam

4044

In [681]:
skipList.nivel_atual

46

In [682]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, skipList.tam)
  skipList.delete(z)

In [683]:
skipList.tam

2962

In [684]:
skipList.nivel_atual

44

In [685]:
imprime_linear(skipList)

{'task_id': 0, 'start_time': 64373448.79108468, 'end_time': 64373548.79108468}
{'task_id': 1, 'start_time': 474218645.0512261, 'end_time': 474218845.0512261}
{'task_id': 3, 'start_time': 26571211.8909201, 'end_time': 26571311.8909201}
{'task_id': 4, 'start_time': 298363689.6643726, 'end_time': 298364089.6643726}
{'task_id': 5, 'start_time': 45701643.09563986, 'end_time': 45701943.09563986}
{'task_id': 6, 'start_time': 16906717.39182673, 'end_time': 16906817.39182673}
{'task_id': 7, 'start_time': 33329449.019334782, 'end_time': 33329549.019334782}
{'task_id': 9, 'start_time': 111384861.80051246, 'end_time': 111385361.80051246}
{'task_id': 14, 'start_time': 68278494.99700505, 'end_time': 68278594.99700505}
{'task_id': 16, 'start_time': 4596766.699012156, 'end_time': 4596866.699012156}
{'task_id': 17, 'start_time': 32206522.764057282, 'end_time': 32206922.764057282}
{'task_id': 18, 'start_time': 70401334.85755463, 'end_time': 70401434.85755463}
{'task_id': 19, 'start_time': 49778851.90200

In [686]:
imprime_estrutura(skipList)

--- Nível 44 ---
407 -> 1044 -> 3511 -> NULL
--- Nível 43 ---
407 -> 1044 -> 2668 -> 3511 -> NULL
--- Nível 42 ---
407 -> 1044 -> 2668 -> 3511 -> 3985 -> NULL
--- Nível 41 ---
407 -> 1044 -> 2668 -> 2695 -> 3511 -> 3985 -> NULL
--- Nível 40 ---
206 -> 364 -> 407 -> 1044 -> 1695 -> 2391 -> 2668 -> 2695 -> 3511 -> 3985 -> 4009 -> NULL
--- Nível 39 ---
206 -> 364 -> 407 -> 1044 -> 1695 -> 2391 -> 2668 -> 2695 -> 3511 -> 3871 -> 3985 -> 4009 -> NULL
--- Nível 38 ---
206 -> 364 -> 407 -> 1044 -> 1695 -> 2391 -> 2668 -> 2695 -> 3095 -> 3511 -> 3871 -> 3985 -> 4009 -> NULL
--- Nível 37 ---
206 -> 364 -> 407 -> 1044 -> 1531 -> 1695 -> 2391 -> 2668 -> 2695 -> 3095 -> 3511 -> 3871 -> 3985 -> 3991 -> 4009 -> NULL
--- Nível 36 ---
206 -> 364 -> 407 -> 1044 -> 1531 -> 1619 -> 1695 -> 2391 -> 2668 -> 2695 -> 3095 -> 3511 -> 3786 -> 3871 -> 3985 -> 3991 -> 4009 -> NULL
--- Nível 35 ---
206 -> 364 -> 407 -> 1044 -> 1531 -> 1619 -> 1695 -> 2160 -> 2391 -> 2459 -> 2668 -> 2695 -> 2801 -> 3095 -> 3230 ->

### Search

In [687]:
for i in range(skipList.tam):
  z = np.random.randint(0, skipList.tam)
  if not skipList.search(z):
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {skipList.search(z)}\n=============================================================')

{'task_id': 845, 'start_time': 10026788.492548343, 'end_time': 10026888.492548343}
Chave:1039; Resultado da Busca: False
{'task_id': 2481, 'start_time': 97639982.140701, 'end_time': 97640082.140701}
{'task_id': 571, 'start_time': 50734377.37712544, 'end_time': 50734477.37712544}
{'task_id': 668, 'start_time': 215903621.91573042, 'end_time': 215903921.91573042}
Chave:2730; Resultado da Busca: False
{'task_id': 1118, 'start_time': 6734626.782128139, 'end_time': 6734726.782128139}
{'task_id': 707, 'start_time': 35440215.92377622, 'end_time': 35440515.92377622}
{'task_id': 2091, 'start_time': 232281366.77762276, 'end_time': 232281566.77762276}
{'task_id': 2256, 'start_time': 5478029.538908019, 'end_time': 5478129.538908019}
Chave:212; Resultado da Busca: False
{'task_id': 1517, 'start_time': 23033208.298720133, 'end_time': 23033308.298720133}
{'task_id': 85, 'start_time': 97307274.62264083, 'end_time': 97307674.62264083}
{'task_id': 2430, 'start_time': 31812995.06412548, 'end_time': 318130

### SEARCH INTERVAL

In [688]:
for i in range(skipList.tam):
  a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
  b = np.random.randint(0, N*np.random.randint(1,N))
  while a > b:
      a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
      b = np.random.randint(0, N*np.random.randint(1,N))
  r = (skipList.searchinterval(a,b))
  print(f'=============================\nA={a}; B={b}')
  for k in r:
    if k is not None:
      print(k.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
{'task_id': 3098, 'start_time': 5028967.304571181, 'end_time': 5029067.304571181}
{'task_id': 3113, 'start_time': 3203141.042687481, 'end_time': 3203241.042687481}
{'task_id': 3117, 'start_time': 7515236.977655153, 'end_time': 7515836.977655153}
{'task_id': 3133, 'start_time': 12113028.247035006, 'end_time': 12113128.247035006}
{'task_id': 3149, 'start_time': 9176999.901313629, 'end_time': 9177099.901313629}
{'task_id': 3159, 'start_time': 9784797.21245583, 'end_time': 9785197.21245583}
{'task_id': 3160, 'start_time': 9090373.061442515, 'end_time': 9090473.061442515}
{'task_id': 3170, 'start_time': 15750543.282465924, 'end_time': 15753043.282465924}
{'task_id': 3190, 'start_time': 8140083.637898436, 'end_time': 8140983.637898436}
{'task_id': 3192, 'start_time': 6573106.601015834, 'end_time': 6573206.601015834}
{'task_id': 3198, 'start_time': 1717559.8978820147, 'end_time': 1717659.8978820147}
{'task_id': 3219, 'start_time': 265

## Distribuição Uniforme - AVL

### Instanciação

In [689]:
raiz=None
arvoreAVL = AVL()

### INSERT

In [690]:
t = generate_uniform_tasks(N)
for k in t:
  m = Task(k)
  raiz = arvoreAVL.insert(raiz, m)

In [691]:
percursoIN_ORDER(raiz)

{'task_id': 0, 'start_time': 386106, 'end_time': 538786}
{'task_id': 1, 'start_time': 700925, 'end_time': 788975}
{'task_id': 2, 'start_time': 138910, 'end_time': 977571}
{'task_id': 3, 'start_time': 266887, 'end_time': 447247}
{'task_id': 4, 'start_time': 27136, 'end_time': 608731}
{'task_id': 5, 'start_time': 781249, 'end_time': 972336}
{'task_id': 6, 'start_time': 259650, 'end_time': 929079}
{'task_id': 7, 'start_time': 229508, 'end_time': 653486}
{'task_id': 8, 'start_time': 306262, 'end_time': 685716}
{'task_id': 9, 'start_time': 266776, 'end_time': 896792}
{'task_id': 10, 'start_time': 435155, 'end_time': 716362}
{'task_id': 11, 'start_time': 215839, 'end_time': 592420}
{'task_id': 12, 'start_time': 187092, 'end_time': 337024}
{'task_id': 13, 'start_time': 871608, 'end_time': 941687}
{'task_id': 14, 'start_time': 107583, 'end_time': 507924}
{'task_id': 15, 'start_time': 895113, 'end_time': 927143}
{'task_id': 16, 'start_time': 828298, 'end_time': 997724}
{'task_id': 17, 'start_ti

### DELETE

In [692]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, random.randint(1,N))
  raiz = arvoreAVL.delete(raiz, z)

Removendo Nó de chave: 382
Removendo Nó de chave: 551
Removendo Nó de chave: 550
Removendo Nó de chave: 128
Removendo Nó de chave: 141
Removendo Nó de chave: 140
Removendo Nó de chave: 1335
Removendo Nó de chave: 1334
Removendo Nó de chave: 205
Removendo Nó de chave: 204
Removendo Nó de chave: 4
Removendo Nó de chave: 71
Removendo Nó de chave: 70
Removendo Nó de chave: 222
Removendo Nó de chave: 505
Removendo Nó de chave: 504
Removendo Nó de chave: 1166
Removendo Nó de chave: 184
Removendo Nó de chave: 593
Removendo Nó de chave: 592
Removendo Nó de chave: 1902
Removendo Nó de chave: 1089
Removendo Nó de chave: 1088
Removendo Nó de chave: 2045
Removendo Nó de chave: 2044
Removendo Nó de chave: 550
Removendo Nó de chave: 549
Removendo Nó de chave: 822
Removendo Nó de chave: 902
Removendo Nó de chave: 8
Removendo Nó de chave: 298
Removendo Nó de chave: 3006
Removendo Nó de chave: 563
Removendo Nó de chave: 562
Removendo Nó de chave: 983
Removendo Nó de chave: 982
Removendo Nó de chave: 29

In [693]:
percursoIN_ORDER(raiz)

{'task_id': 7, 'start_time': 229508, 'end_time': 653486}
{'task_id': 9, 'start_time': 266776, 'end_time': 896792}
{'task_id': 33, 'start_time': 356804, 'end_time': 481296}
{'task_id': 42, 'start_time': 420878, 'end_time': 641295}
{'task_id': 47, 'start_time': 202193, 'end_time': 530680}
{'task_id': 62, 'start_time': 235861, 'end_time': 857619}
{'task_id': 70, 'start_time': 684414, 'end_time': 799329}
{'task_id': 73, 'start_time': 52536, 'end_time': 277834}
{'task_id': 87, 'start_time': 49176, 'end_time': 738496}
{'task_id': 90, 'start_time': 789666, 'end_time': 962550}
{'task_id': 95, 'start_time': 834877, 'end_time': 873178}
{'task_id': 116, 'start_time': 469650, 'end_time': 983976}
{'task_id': 130, 'start_time': 71733, 'end_time': 175970}
{'task_id': 136, 'start_time': 254436, 'end_time': 694460}
{'task_id': 137, 'start_time': 103565, 'end_time': 709429}
{'task_id': 152, 'start_time': 384020, 'end_time': 440488}
{'task_id': 160, 'start_time': 161948, 'end_time': 397448}
{'task_id': 1

### Search

In [694]:
for i in range(floor(N/2)):
  z = np.random.randint(0, np.random.randint(1,N))
  r = arvoreAVL.search(raiz, z)
  if r:
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {r.conteudo.atributos}\n=============================================================')

Chave:744; Resultado da Busca: {'task_id': 744, 'start_time': 134137, 'end_time': 363499}
Chave:980; Resultado da Busca: {'task_id': 980, 'start_time': 379727, 'end_time': 986303}
Chave:1363; Resultado da Busca: {'task_id': 1363, 'start_time': 241675, 'end_time': 662229}
Chave:1390; Resultado da Busca: {'task_id': 1390, 'start_time': 239156, 'end_time': 448573}
Chave:2404; Resultado da Busca: {'task_id': 2404, 'start_time': 388121, 'end_time': 757875}
Chave:618; Resultado da Busca: {'task_id': 618, 'start_time': 459169, 'end_time': 539143}
Chave:367; Resultado da Busca: {'task_id': 367, 'start_time': 181357, 'end_time': 367849}
Chave:1233; Resultado da Busca: {'task_id': 1233, 'start_time': 459975, 'end_time': 894201}
Chave:1306; Resultado da Busca: {'task_id': 1306, 'start_time': 542461, 'end_time': 860248}
Chave:2654; Resultado da Busca: {'task_id': 2654, 'start_time': 857555, 'end_time': 987957}
Chave:2248; Resultado da Busca: {'task_id': 2248, 'start_time': 272168, 'end_time': 5467

### SEARCH INTERVAL

In [695]:
for i in range(N):
  a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
  b = np.random.randint(0, N*np.random.randint(1,N))
  while a > b:
      a = np.random.randint(1, floor(N/2)*np.random.randint(1,floor(N/2)))
      b = np.random.randint(0, N*np.random.randint(1,N))
  out = []
  r = arvoreAVL.searchinterval(raiz,a,b,out)
  print(f'=============================\nA={a}; B={b}')
  for u in r:
    print(u.conteudo.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
{'task_id': 2612, 'start_time': 184511, 'end_time': 526266}
{'task_id': 2611, 'start_time': 465163, 'end_time': 591381}
{'task_id': 2608, 'start_time': 219462, 'end_time': 290453}
{'task_id': 2607, 'start_time': 323812, 'end_time': 993214}
{'task_id': 2606, 'start_time': 703976, 'end_time': 836783}
{'task_id': 2605, 'start_time': 56660, 'end_time': 533792}
{'task_id': 2604, 'start_time': 598433, 'end_time': 673722}
{'task_id': 2602, 'start_time': 65203, 'end_time': 128317}
{'task_id': 2600, 'start_time': 345842, 'end_time': 879474}
{'task_id': 2598, 'start_time': 323361, 'end_time': 698895}
{'task_id': 2597, 'start_time': 339423, 'end_time': 606272}
{'task_id': 2596, 'start_time': 492259, 'end_time': 738361}
{'task_id': 2595, 'start_time': 460313, 'end_time': 848196}
{'task_id': 2593, 'start_time': 221057, 'end_time': 438899}
{'task_id': 2592, 'start_time': 146909, 'end_time': 595925}
{'task_id': 2591, 'start_time': 351799, 'en

## Distribuição Normal - AVL

### Instanciação

In [696]:
arvoreAVL = AVL()

### INSERT

In [697]:
t = generate_normal_tasks(N)
raiz = None
for k in t:
  k['start_time'] = k['start_time'].item()
  k['end_time'] = k['end_time'].item()
  m = Task(k)
  raiz = arvoreAVL.insert(raiz, m)

In [698]:
percursoIN_ORDER(raiz)

{'task_id': 0, 'start_time': 511882352.56497735, 'end_time': 512712800.07926434}
{'task_id': 1, 'start_time': 604469104.3967127, 'end_time': 604949956.4459503}
{'task_id': 2, 'start_time': 662233039.4988893, 'end_time': 663124777.6911299}
{'task_id': 3, 'start_time': 425845640.96819466, 'end_time': 425870567.89707506}
{'task_id': 4, 'start_time': 511237419.6228536, 'end_time': 511402671.9419429}
{'task_id': 5, 'start_time': 436902387.00447214, 'end_time': 437024020.6632256}
{'task_id': 6, 'start_time': 305971012.70584047, 'end_time': 306598163.4717557}
{'task_id': 7, 'start_time': 331654051.63787293, 'end_time': 332229002.09824324}
{'task_id': 8, 'start_time': 437410115.6114216, 'end_time': 437413527.3538985}
{'task_id': 9, 'start_time': 523158311.5559802, 'end_time': 523718538.2780959}
{'task_id': 10, 'start_time': 433015976.71364117, 'end_time': 433433224.114073}
{'task_id': 11, 'start_time': 418835713.9863911, 'end_time': 419586436.60560983}
{'task_id': 12, 'start_time': 590477504.6

### DELETE

In [699]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, random.randint(1,N))
  raiz = arvoreAVL.delete(raiz, z)

Removendo Nó de chave: 3
Removendo Nó de chave: 2
Removendo Nó de chave: 1426
Removendo Nó de chave: 608
Removendo Nó de chave: 2551
Removendo Nó de chave: 2550
Removendo Nó de chave: 1082
Removendo Nó de chave: 97
Removendo Nó de chave: 96
Removendo Nó de chave: 24
Removendo Nó de chave: 2772
Removendo Nó de chave: 1308
Removendo Nó de chave: 2706
Removendo Nó de chave: 1719
Removendo Nó de chave: 1718
Removendo Nó de chave: 1462
Removendo Nó de chave: 336
Removendo Nó de chave: 906
Removendo Nó de chave: 1432
Removendo Nó de chave: 1914
Removendo Nó de chave: 3558
Removendo Nó de chave: 12
Removendo Nó de chave: 2746
Removendo Nó de chave: 622
Removendo Nó de chave: 1518
Removendo Nó de chave: 112
Removendo Nó de chave: 128
Removendo Nó de chave: 900
Removendo Nó de chave: 100
Removendo Nó de chave: 1679
Removendo Nó de chave: 1678
Removendo Nó de chave: 1554
Removendo Nó de chave: 881
Removendo Nó de chave: 880
Removendo Nó de chave: 1528
Removendo Nó de chave: 570
Removendo Nó de c

In [700]:
percursoIN_ORDER(raiz)

{'task_id': 2, 'start_time': 662233039.4988893, 'end_time': 663124777.6911299}
{'task_id': 5, 'start_time': 436902387.00447214, 'end_time': 437024020.6632256}
{'task_id': 6, 'start_time': 305971012.70584047, 'end_time': 306598163.4717557}
{'task_id': 7, 'start_time': 331654051.63787293, 'end_time': 332229002.09824324}
{'task_id': 8, 'start_time': 437410115.6114216, 'end_time': 437413527.3538985}
{'task_id': 9, 'start_time': 523158311.5559802, 'end_time': 523718538.2780959}
{'task_id': 10, 'start_time': 433015976.71364117, 'end_time': 433433224.114073}
{'task_id': 11, 'start_time': 418835713.9863911, 'end_time': 419586436.60560983}
{'task_id': 13, 'start_time': 465717363.4474688, 'end_time': 466161899.62943596}
{'task_id': 14, 'start_time': 410916036.2302726, 'end_time': 411445017.8586616}
{'task_id': 16, 'start_time': 420581435.08916664, 'end_time': 420818150.79134136}
{'task_id': 17, 'start_time': 544099708.996726, 'end_time': 544421101.5027883}
{'task_id': 18, 'start_time': 412555998

### Search

In [701]:
for i in range(floor(N/2)):
  z = np.random.randint(0, np.random.randint(1,N))
  r = arvoreAVL.search(raiz, z)
  if r:
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {r.conteudo.atributos}\n=============================================================')

A saída de streaming foi truncada nas últimas 5000 linhas.
Chave:3198; Resultado da Busca: {'task_id': 3198, 'start_time': 579536159.0761516, 'end_time': 580298593.441886}
Chave:1103; Resultado da Busca: {'task_id': 1103, 'start_time': 590520848.0909787, 'end_time': 590921491.4489548}
Chave:528; Resultado da Busca: {'task_id': 528, 'start_time': 303882999.2866651, 'end_time': 304876744.40090024}
Chave:902; Resultado da Busca: {'task_id': 902, 'start_time': 481449158.89890623, 'end_time': 481630588.79275715}
Chave:302; Resultado da Busca: {'task_id': 302, 'start_time': 576654793.096944, 'end_time': 577051332.522933}
Chave:11; Resultado da Busca: {'task_id': 11, 'start_time': 418835713.9863911, 'end_time': 419586436.60560983}
Chave:494; Resultado da Busca: {'task_id': 494, 'start_time': 517326269.2004483, 'end_time': 517345349.62528557}
Chave:770; Resultado da Busca: {'task_id': 770, 'start_time': 547282151.0467987, 'end_time': 547533488.9136211}
Chave:263; Resultado da Busca: {'task_id'

### SEARCH INTERVAL

In [702]:
for i in range(N):
  a = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  b = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  while a > b:
    a = np.random.randint(N, N*np.random.randint(floor(N/2),N))
    b = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  out = []
  r = arvoreAVL.searchinterval(raiz,a,b,out)
  print(f'=============================\nA={a}; B={b}')
  for u in r:
    print(u.conteudo.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
A=2694095; B=8681397
A=6500541; B=7322669
A=2254352; B=4436458
A=5940924; B=6075329
A=3693618; B=10241987
A=28425; B=9162877
A=4693010; B=8327293
A=3837873; B=5213545
A=7761556; B=8390657
A=4833329; B=7894696
A=3080664; B=13101075
A=2705346; B=11229800
A=8084701; B=8458675
A=1983635; B=11579566
A=8104892; B=9610761
A=10310773; B=11961178
A=1832928; B=11767646
A=291407; B=1195154
A=2329951; B=4281530
A=3171950; B=11667023
A=3637299; B=11891649
A=75063; B=12949031
A=6297981; B=8231145
A=8052740; B=8366974
A=968134; B=1411086
A=1019649; B=11420136
A=3728341; B=10182267
A=104138; B=2195530
A=6667528; B=8001551
A=11103080; B=11734623
A=451340; B=10055617
A=4882723; B=8118860
A=6840796; B=8540163
A=2942740; B=5888878
A=7392447; B=11347049
A=536375; B=4256543
A=6576360; B=6944922
A=1049152; B=9627693
A=93886; B=7760085
A=2381037; B=5344048
A=6957738; B=10574383
A=5536905; B=5822372
A=8238368; B=8669052
A=5781727; B=10614088
A=10978214

## Distribuição ZIPF - AVL

### Instanciação

In [703]:
arvoreAVL = AVL()

### INSERT

In [704]:
t = generate_zipf_tasks(N)
raiz = None
for k in t:
  m = Task(k)
  raiz = arvoreAVL.insert(raiz, m)

In [705]:
percursoIN_ORDER(raiz)

{'task_id': 0, 'start_time': 87736480.15677981, 'end_time': 87736580.15677981}
{'task_id': 1, 'start_time': 338778775.67984223, 'end_time': 338778875.67984223}
{'task_id': 2, 'start_time': 47956995.74908623, 'end_time': 47957095.74908623}
{'task_id': 3, 'start_time': 3889479.997314831, 'end_time': 3889579.997314831}
{'task_id': 4, 'start_time': 39439811.90297703, 'end_time': 39439911.90297703}
{'task_id': 5, 'start_time': 24513932.456272613, 'end_time': 24514132.456272613}
{'task_id': 6, 'start_time': 22520126.01442799, 'end_time': 22522226.01442799}
{'task_id': 7, 'start_time': 118396912.4168496, 'end_time': 118397012.4168496}
{'task_id': 8, 'start_time': 326375725.8688761, 'end_time': 326375825.8688761}
{'task_id': 9, 'start_time': 107764490.75167334, 'end_time': 107764590.75167334}
{'task_id': 10, 'start_time': 3142270.801044375, 'end_time': 3143170.801044375}
{'task_id': 11, 'start_time': 90883618.21586762, 'end_time': 90883918.21586762}
{'task_id': 12, 'start_time': 84309297.71678

### DELETE

In [706]:
for i in range(np.random.randint(floor(N))):
  z = np.random.randint(0, random.randint(1,N))
  raiz = arvoreAVL.delete(raiz, z)

Removendo Nó de chave: 593
Removendo Nó de chave: 592
Removendo Nó de chave: 773
Removendo Nó de chave: 772
Removendo Nó de chave: 1847
Removendo Nó de chave: 1846
Removendo Nó de chave: 391
Removendo Nó de chave: 390
Removendo Nó de chave: 315
Removendo Nó de chave: 314
Removendo Nó de chave: 1233
Removendo Nó de chave: 1232
Removendo Nó de chave: 280
Removendo Nó de chave: 1350
Removendo Nó de chave: 324
Removendo Nó de chave: 248
Removendo Nó de chave: 77
Removendo Nó de chave: 76
Removendo Nó de chave: 491
Removendo Nó de chave: 490
Removendo Nó de chave: 99
Removendo Nó de chave: 98
Removendo Nó de chave: 799
Removendo Nó de chave: 798
Removendo Nó de chave: 555
Removendo Nó de chave: 554
Removendo Nó de chave: 730
Removendo Nó de chave: 578
Removendo Nó de chave: 747
Removendo Nó de chave: 746
Removendo Nó de chave: 1349
Removendo Nó de chave: 698
Removendo Nó de chave: 674
Removendo Nó de chave: 771
Removendo Nó de chave: 770
Removendo Nó de chave: 210
Removendo Nó de chave: 123

In [707]:
percursoIN_ORDER(raiz)

{'task_id': 17, 'start_time': 103544900.69862545, 'end_time': 103545000.69862545}
{'task_id': 18, 'start_time': 347431296.32659316, 'end_time': 347431396.32659316}
{'task_id': 21, 'start_time': 124271443.26769775, 'end_time': 124271543.26769775}
{'task_id': 33, 'start_time': 33748802.5468097, 'end_time': 33757802.5468097}
{'task_id': 46, 'start_time': 4038237.5928650135, 'end_time': 4038337.5928650135}
{'task_id': 52, 'start_time': 20701065.81541648, 'end_time': 20701165.81541648}
{'task_id': 53, 'start_time': 2994465.6053970084, 'end_time': 2995065.6053970084}
{'task_id': 54, 'start_time': 22415260.594158933, 'end_time': 22415360.594158933}
{'task_id': 59, 'start_time': 175066744.99381444, 'end_time': 175067044.99381444}
{'task_id': 62, 'start_time': 44600019.61366588, 'end_time': 44600119.61366588}
{'task_id': 73, 'start_time': 76394085.64264996, 'end_time': 76394185.64264996}
{'task_id': 76, 'start_time': 992468.8689490504, 'end_time': 992868.8689490504}
{'task_id': 86, 'start_time'

### Search

In [708]:
for i in range(floor(N/2)):
  z = np.random.randint(0, np.random.randint(1,N))
  r = arvoreAVL.search(raiz, z)
  if r:
    print(f'=============================================================\nChave:{z}; Resultado da Busca: {r.conteudo.atributos}\n=============================================================')

Chave:2511; Resultado da Busca: {'task_id': 2511, 'start_time': 47696306.86804937, 'end_time': 47696406.86804937}
Chave:789; Resultado da Busca: {'task_id': 789, 'start_time': 21607636.262411833, 'end_time': 21607736.262411833}
Chave:1055; Resultado da Busca: {'task_id': 1055, 'start_time': 91243833.59039278, 'end_time': 91243933.59039278}
Chave:361; Resultado da Busca: {'task_id': 361, 'start_time': 90427903.56048007, 'end_time': 90428003.56048007}
Chave:2689; Resultado da Busca: {'task_id': 2689, 'start_time': 54146980.29315641, 'end_time': 54157680.29315641}
Chave:941; Resultado da Busca: {'task_id': 941, 'start_time': 64614353.724441506, 'end_time': 64614453.724441506}
Chave:919; Resultado da Busca: {'task_id': 919, 'start_time': 144762751.08540133, 'end_time': 144763151.08540133}
Chave:1696; Resultado da Busca: {'task_id': 1696, 'start_time': 57991046.825421385, 'end_time': 57991246.825421385}
Chave:3822; Resultado da Busca: {'task_id': 3822, 'start_time': 303942195.02142435, 'end

### SEARCH INTERVAL

In [709]:
for i in range(N):
  a = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  b = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  while a > b:
    a = np.random.randint(N, N*np.random.randint(floor(N/2),N))
    b = np.random.randint(N, N*np.random.randint(floor(N/2),N))
  out = []
  r = arvoreAVL.searchinterval(raiz,a,b,out)
  print(f'=============================\nA={a}; B={b}')
  for u in r:
    print(u.conteudo.atributos)
  print('=============================')

A saída de streaming foi truncada nas últimas 5000 linhas.
A=631109; B=3137811
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time': 1341062.02072469}
A=1468356; B=4480361
A=4198080; B=10476739
A=5060130; B=10214312
A=8840034; B=11260567
A=1059405; B=12984788
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time': 1341062.02072469}
A=2711475; B=5462591
A=380849; B=4494799
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time': 1341062.02072469}
A=547779; B=9698759
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time': 1341062.02072469}
A=8082256; B=8224927
A=7221610; B=14400490
A=3629005; B=8358927
A=3440013; B=5448323
A=1910209; B=14456486
A=785472; B=4049846
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time': 1341062.02072469}
A=1022552; B=5446019
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time': 1341062.02072469}
A=3717985; B=10028550
A=1434747; B=6772522
A=1121987; B=1834442
{'task_id': 4031, 'start_time': 1340962.02072469, 'end_time